# Deep Learning Finals Notebook: Pixels to Prediction (Kaggle)

Author: Thomas Kong

NetId: tk2558

# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct`

**Fine-Tuning:** QLoRA (4-bit NF4)

**Scoring:** Multiple-choice log-likelihood

---

### **Section 0: Installing Necessary Packages**

> Make sure all packages and their versions are available and correct

> Uncomment whatever is needed to install for notebook environment

> Make sure train.csv, val.csv and test.csv is uploaded for notebook to access

In [ ]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow

### **Section 1: Imports & Configuration**

> Initialize variables for the notebook and models

In [ ]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
import math
import time

from pathlib import Path
from typing import List, Optional

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("/kaggle/input/datasets/tk2558/science-data/Data")           # KAGGLE
OUTPUT_DIR = Path("/kaggle/working/outputs")  # KAGGLE

OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR  = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(exist_ok=True)

# ── Hyper-parameters ──────────────────────────────────────────────────────────
IMG_SIZE        = 384   # 224
BATCH_SIZE      = 4
TRAIN_BATCH     = 2
GRAD_ACCUM      = 8
NUM_EPOCHS      = 2     # 2
LR              = 2e-4
LORA_R          = 7     # 8
LORA_ALPHA      = 14    # 16
LORA_DROPOUT    = 0.05
MAX_SEQ_LEN     = 2048  #512

CHOICE_LETTERS  = "ABCDE"

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

### **Section 2: Load and Preprocess Data**

> Load Datasets for Training, Validation and Testing

In [ ]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

In [ ]:
# ── Training Size Control ─────────────────────────────────────────────────
# Adjust TRAIN_FRACTION (0.0–1.0) or TRAIN_MAX_SAMPLES to trade off

TRAIN_FRACTION    = 0.25    # ← change this (0.0–1.0)
TRAIN_MAX_SAMPLES = None   # ← or set a hard cap, e.g. 500; None = use fraction only

def subsample_train(df: pd.DataFrame, fraction: float = 1.0, max_samples: int = None, seed: int = SEED) -> pd.DataFrame:
    if fraction <= 0 or fraction > 1: # EDGE CASE SAFETY
        raise ValueError("TRAIN_FRACTION must be in (0, 1]")

    n_target = int(len(df) * fraction)
    if max_samples is not None:
        n_target = min(n_target, max_samples)

    if n_target >= len(df):
        print(f"Using full training set: {len(df):,} rows")
        return df.reset_index(drop=True)

    # Stratified sample — preserve answer-label distribution
    sampled = (
        df.groupby("answer", group_keys=False)
        .apply(lambda g: g.sample(
            n=max(1, round(n_target * len(g) / len(df))),
            random_state=seed,
        ))
        .reset_index(drop=True)
    )

    print(f"Subsampled training set: {len(sampled):,} / {len(df):,} rows "f"({len(sampled)/len(df)*100:.1f}%)")
    print(f"Answer distribution:\n"f"{sampled['answer'].value_counts(normalize=True).sort_index().round(3)}")
    return sampled

train_df = subsample_train(train_df, fraction=TRAIN_FRACTION, max_samples=TRAIN_MAX_SAMPLES)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# VISUALIZE DATASET GRAPHS
TRAIN_COLOR = "#3498db"   # blue  = train
VAL_COLOR   = "#e67e22"   # orange = val
OVERALL_COLOR = "black"

def add_value_labels(ax, bars, fmt="{:.0%}", offset=0.012, fontsize=8):
    """Annotate each bar with its value."""
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + offset,
                fmt.format(h), ha="center", va="bottom", fontsize=fontsize)

def add_hvalue_labels(ax, bars, fmt="{:,}", offset=8, fontsize=8):
    """Annotate each horizontal bar with its value."""
    for bar in bars:
        w = bar.get_width()
        ax.text(w + offset, bar.get_y() + bar.get_height() / 2,
                fmt.format(int(w)), ha="left", va="center", fontsize=fontsize)

# Split sizes
fig, ax = plt.subplots(figsize=(6, 4))
splits = ["Train", "Validation"]
counts = [len(train_df), len(val_df)]
colors = [TRAIN_COLOR, VAL_COLOR]
bars   = ax.bar(splits, counts, color=colors, width=0.4)
for bar, n in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 30,
            f"{n:,}", ha="center", va="bottom", fontsize=11, fontweight="bold")
ax.set_title("Dataset split sizes", fontsize=13, pad=12)
ax.set_ylabel("Number of examples")
ax.set_ylim(0, max(counts) * 1.15)
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.tick_params(axis='x', labelsize=11)
plt.tight_layout()
plt.savefig("dataset_split_sizes.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → dataset_split_sizes.png")

In [ ]:
# Subject distribution
fig, ax = plt.subplots(figsize=(8, 4))
subjects = sorted(set(train_df["subject"].unique()) | set(val_df["subject"].unique()))
train_subj = train_df["subject"].value_counts(normalize=True).reindex(subjects, fill_value=0)
val_subj = val_df["subject"].value_counts(normalize=True).reindex(subjects, fill_value=0)
x = np.arange(len(subjects))
width = 0.35
b1 = ax.bar(x - width/2, train_subj.values, width, label="Train", color=TRAIN_COLOR)
b2 = ax.bar(x + width/2, val_subj.values,   width, label="Validation", color=VAL_COLOR)
add_value_labels(ax, b1)
add_value_labels(ax, b2)
ax.set_title("Subject distribution — train vs validation", fontsize=13, pad=12)
ax.set_ylabel("Proportion of split")
ax.set_xticks(x)
ax.set_xticklabels(subjects, fontsize=9)
ax.set_ylim(0, max(train_subj.max(), val_subj.max()) + 0.12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("dataset_subject_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → dataset_subject_dist.png")

In [ ]:
# Grade distribution
fig, ax = plt.subplots(figsize=(11, 4))
all_grades = sorted(set(train_df["grade"].unique()) | set(val_df["grade"].unique()), key=lambda g: int(g.replace("grade", "")))
grade_labels = [g.replace("grade", "Grade ") for g in all_grades]
train_grade = train_df["grade"].value_counts(normalize=True).reindex(all_grades, fill_value=0)
val_grade = val_df["grade"].value_counts(normalize=True).reindex(all_grades, fill_value=0)
x = np.arange(len(all_grades))
width = 0.35
b1 = ax.bar(x - width/2, train_grade.values, width, label="Train", color=TRAIN_COLOR)
b2 = ax.bar(x + width/2, val_grade.values,   width, label="Validation", color=VAL_COLOR)
add_value_labels(ax, b1, fontsize=7)
add_value_labels(ax, b2, fontsize=7)
ax.set_title("Grade distribution — train vs validation", fontsize=13, pad=12)
ax.set_ylabel("Proportion of split")
ax.set_xticks(x)
ax.set_xticklabels(grade_labels, rotation=30, ha="right", fontsize=9)
ax.set_ylim(0, max(train_grade.max(), val_grade.max()) + 0.1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("dataset_grade_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → dataset_grade_dist.png")

In [ ]:
# Number of choices distribution
fig, ax = plt.subplots(figsize=(8, 4))
all_nc = sorted(set(train_df["num_choices"].unique()) | set(val_df["num_choices"].unique()))
nc_labels = [f"{n} choices" for n in all_nc]
train_nc = train_df["num_choices"].value_counts(normalize=True).reindex(all_nc, fill_value=0)
val_nc = val_df["num_choices"].value_counts(normalize=True).reindex(all_nc, fill_value=0)
x = np.arange(len(all_nc))
width = 0.35
b1 = ax.bar(x - width/2, train_nc.values, width, label="Train", color=TRAIN_COLOR)
b2 = ax.bar(x + width/2, val_nc.values,   width, label="Validation", color=VAL_COLOR)
add_value_labels(ax, b1)
add_value_labels(ax, b2)
ax.set_title("Number of choices distribution — train vs validation", fontsize=13, pad=12)
ax.set_ylabel("Proportion of split")
ax.set_xticks(x)
ax.set_xticklabels(nc_labels, fontsize=10)
ax.set_ylim(0, max(train_nc.max(), val_nc.max()) + 0.12)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("dataset_num_choices_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → dataset_num_choices_dist.png")

In [ ]:
# Answer label distribution
fig, ax = plt.subplots(figsize=(8, 4))
all_ans = sorted(set(train_df["answer"].unique()) | set(val_df["answer"].unique()))
ans_labels = [CHOICE_LETTERS[i] for i in all_ans]
train_ans = train_df["answer"].value_counts(normalize=True).reindex(all_ans, fill_value=0)
val_ans = val_df["answer"].value_counts(normalize=True).reindex(all_ans, fill_value=0)
x = np.arange(len(all_ans))
width = 0.35
b1 = ax.bar(x - width/2, train_ans.values, width, label="Train", color=TRAIN_COLOR)
b2 = ax.bar(x + width/2, val_ans.values,   width, label="Validation", color=VAL_COLOR)
add_value_labels(ax, b1)
add_value_labels(ax, b2)

uniform = 1 / len(all_ans)
ax.axhline(uniform, color=OVERALL_COLOR, linestyle="--", linewidth=1,label=f"Uniform ({uniform:.0%})")
ax.set_title("Answer label distribution — train vs validation", fontsize=13, pad=12)
ax.set_ylabel("Proportion of split")
ax.set_xticks(x)
ax.set_xticklabels(ans_labels, fontsize=11)
ax.set_ylim(0, max(train_ans.max(), val_ans.max()) + 0.1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("dataset_answer_dist.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved → dataset_answer_dist.png")

### **Section 3: Prompt Engineering and Dataset**

> Normalize Dataset and apply prompt format

> Adjust size of training dataset used to adjust training time

> Check for any missing image files

In [ ]:
# ── 3b. Prompt Engineering ───────────────────────────────────────────────────
def _context_str(row: pd.Series) -> str:
    """Concatenate lecture + hint if present."""
    parts = []
    for field in ["lecture", "hint"]:
        val = row.get(field, "")
        if pd.notna(val) and str(val).strip():
            parts.append(str(val).strip())
    return "\n".join(parts)

def _subject_str(row: pd.Series) -> str:
    parts = []
    for field in ["subject", "topic", "grade"]:
        val = row.get(field, "")
        if pd.notna(val) and str(val).strip():
            label = field.capitalize()
            parts.append(f"{label}: {str(val).strip()}")
    return " | ".join(parts)

def build_messages(row: pd.Series, answer_idx: int = None) -> list:
    ctx = _context_str(row)
    sub = _subject_str(row)
    choices = row["choices"]
    choices_str = "\n".join(f"  {CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    user_text = ""
    if sub:
        user_text += f"{sub}\n\n"
    if ctx:
        user_text += f"Context:\n{ctx}\n\n"

    user_text += f"Question: {row['question']}\nChoices:\n{choices_str}\n"
    user_text += "Answer with a single letter only (e.g. A)."

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": user_text},
            ],
        }
    ]

    if answer_idx is not None:
        choice_text = choices[answer_idx]
        answer_text = (
            f"Looking at the image and the context provided, "
            f"the best answer is {CHOICE_LETTERS[answer_idx]}. {choice_text}"
        )
        messages.append({"role": "assistant", "content": [{"type": "text", "text": answer_text}]})

    return messages

# Quick visual check
print(json.dumps(build_messages(train_df.iloc[0]), indent=2))

In [ ]:
# ── 3c. PyTorch Dataset ───────────────────────────────────────────────────────
def load_image(path, size: int = IMG_SIZE) -> Image.Image:
    """
    Resize to fit within (size x size) while preserving aspect ratio,
    then pad with neutral gray to reach exactly (size x size).
    """
    img = Image.open(path).convert("RGB")
    img.thumbnail((size, size), Image.BICUBIC)   # shrinks in-place, keeps ratio
    if img.size == (size, size):
        return img

    padded = Image.new("RGB", (size, size), (127, 127, 127))  # neutral gray pad
    offset_x = (size - img.width)  // 2
    offset_y = (size - img.height) // 2
    padded.paste(img, (offset_x, offset_y))

    return padded

class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 384, is_train: bool = True): # img_size: int = 224
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        # img = Image.open(self.data_dir / rel_path).convert("RGB")
        # img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        # return img
        return load_image(self.data_dir / rel_path, size=self.img_size)

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])
        answer_idx = int(row["answer"]) if "answer" in row and pd.notna(row.get("answer")) else -1

        out = {
            "id":       row["id"],
            "image":    img,
            #"messages": build_messages(row, include_answer=self.is_train),
            "messages": build_messages(row, answer_idx=answer_idx if self.is_train else None),
            "choices":  row["choices"],
            "answer":   int(row["answer"]) if "answer" in row and pd.notna(row.get("answer")) else -1,
        }
        return out

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

In [ ]:
# ── Image path validation ─────────────────────────────────────────────────────
def drop_missing_images(df: pd.DataFrame, data_dir: Path, split_name: str) -> pd.DataFrame:
    mask = df["image_path"].apply(lambda p: (data_dir / p).exists())
    missing = (~mask).sum()
    if missing:
        print(f"[{split_name}] dropping {missing} rows with missing images")
    else:
        print(f"[{split_name}] all {len(df)} images found")

    return df[mask].reset_index(drop=True)

train_df = drop_missing_images(train_df, DATA_DIR, "train")
val_df   = drop_missing_images(val_df,   DATA_DIR, "val")
test_df  = drop_missing_images(test_df,  DATA_DIR, "test")

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

### **Section 4: Model Loading**

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

#### **Section 4A: Secret Key for Hugging Face API**

> Make sure notebook can access the secret key/token and you can assign it to an environment variable.

> If using Kaggle, uncomment top half of codeblock and comment the bottom. If using Google Colab, uncomment bottom half of codeblock and comment the top.

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
user_secrets = UserSecretsClient()

# Set the HF_TOKEN environment variable
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")

# --------------------------------------------------------------------------- #

# from google.colab import userdata
# import os

# # Access the secret and assign it to an environment variable
# # Replace 'MY_API_KEY' with the exact name you used in the Secrets Manager
# api_key = userdata.get("HF_TOKEN")
# os.environ["HF_TOKEN"] = api_key

print("KEY READY")

#### **Section 4B: Load Model**

> Loading and configuring the pre-trained vision-language model. Initialize Processor and 4-bit QLoRA Quantization Configuration.

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
ADAPTER_ID    = "tk2558/DL_Finals_Model"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(ADAPTER_ID, token=os.environ["HF_TOKEN"])
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "right"

base_model = AutoModelForVision2Seq.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    low_cpu_mem_usage=True,
    token=os.environ["HF_TOKEN"],
)

model = PeftModel.from_pretrained(
    base_model,
    ADAPTER_ID,
    token=os.environ["HF_TOKEN"],
)
model.eval()
print("Model ready")

#### **Section 4C:  Log-Likelihood Inference (zero-shot baseline)**

> For each question we score every answer choice by computing the conditional
log-probability of the answer letter (A/B/C/D/E) given the prompt.
The choice with the highest log-prob is selected as the prediction.

> **Process:**
> 1. Iterate through Choices
> 2. Build Messages for each choice
> 3. Prepare Model Inputs: The processor then tokenizes the full_text and prepares the image into inputs suitable for the model
> 4. Label Masking
> 5. Calculate
> 6. Store Scores: The negative of the loss (-outputs.loss.item()) is appended to a list of scores
> 7. Select Best Choice

In [ ]:
@torch.inference_mode()
def score_choices_loglik(model, processor, image: Image.Image, row: pd.Series, device) -> int:
    """
    Single-sample log-likelihood scorer using full choice text.
    Drop-in replacement for the original function.
    """
    choices = row["choices"]
    scores  = []

    for idx in range(len(choices)):
        full_msgs   = build_messages(row, answer_idx=idx)
        prefix_msgs = build_messages(row, answer_idx=None)

        full_text   = processor.apply_chat_template(full_msgs,   tokenize=False, add_generation_prompt=False)
        prefix_text = processor.apply_chat_template(prefix_msgs, tokenize=False, add_generation_prompt=True)

        inputs = processor(
            text=[full_text], images=[image],
            return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
        )
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
                  for k, v in inputs.items()}

        labels = inputs["input_ids"].clone()

        prefix_ids = processor(
            text=[prefix_text], images=[image],
            return_tensors="pt", padding=False,
            truncation=True, max_length=MAX_SEQ_LEN,
        )["input_ids"]
        prefix_len = prefix_ids.shape[1]

        labels[:, :prefix_len] = -100
        outputs = model(**inputs, labels=labels)
        scores.append(-outputs.loss.item())

    return int(np.argmax(scores))

### **Section 8: Test Inference & Submission Generation**

> Prepare for model to start generating outputs (Batch and Single Version)

> We will use Singe Version for Generation

In [ ]:
# ── Quick sanity check on 5 validation examples ───────────────────────────────
print("Sanity check — scoring 5 val examples with log-likelihood...")
model.eval()
correct = 0
for i in range(5):
    row   = val_df.iloc[i]
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    pred  = score_choices_loglik(model, processor, image, row, device)
    gt    = int(row["answer"])
    mark  = "✓" if pred == gt else "✗"
    print(f"  [{mark}] pred={CHOICE_LETTERS[pred]}  gt={CHOICE_LETTERS[gt]}  | {row['question']}")
    correct += int(pred == gt)
print(f"Zero-shot accuracy on 5 samples: {correct}/5")

#### **Section 8A: Batch Version of Log-Likeihood Inference**

In [ ]:
@torch.inference_mode()
def score_choices_loglik_batch(model, processor, images: list, rows: list, device) -> list[int]:
    """
    Batched log-likelihood scorer using full choice text.
    Drop-in replacement for the original function.
    """
    B = len(rows)
    max_choices = max(len(r["choices"]) for r in rows)
    scores = [[float("-inf")] * max_choices for _ in range(B)]

    for c in range(max_choices):
        batch_texts   = []
        batch_images  = []
        valid_indices = []

        for b, (row, img) in enumerate(zip(rows, images)):
            if c >= len(row["choices"]):
                continue

            full_msgs = build_messages(row, answer_idx=c)
            full_text = processor.apply_chat_template(
                full_msgs, tokenize=False, add_generation_prompt=False
            )
            batch_texts.append(full_text)
            batch_images.append(img)
            valid_indices.append(b)

        if not batch_texts:
            continue

        inputs = processor(
            text=batch_texts, images=batch_images,
            return_tensors="pt", padding=True,
            truncation=True, max_length=MAX_SEQ_LEN,
        )
        inputs = {k: v.to(model.device) if torch.is_tensor(v) else v
                  for k, v in inputs.items()}

        labels = inputs["input_ids"].clone()

        for local_b, global_b in enumerate(valid_indices):
            row = rows[global_b]
            prefix_msgs = build_messages(row, answer_idx=None)
            prefix_text = processor.apply_chat_template(
                prefix_msgs, tokenize=False, add_generation_prompt=True
            )
            prefix_ids = processor(
                text=[prefix_text], images=[images[global_b]],
                return_tensors="pt", padding=False,
                truncation=True, max_length=MAX_SEQ_LEN,
            )["input_ids"]
            prefix_len = prefix_ids.shape[1]
            labels[local_b, :prefix_len] = -100

        labels[inputs["input_ids"] == processor.tokenizer.pad_token_id] = -100

        outputs = model(**inputs)
        logits  = outputs.logits

        shift_logits = logits[:, :-1, :].contiguous()
        shift_labels = labels[:, 1:].contiguous()

        loss_fct    = torch.nn.CrossEntropyLoss(reduction="none", ignore_index=-100)
        token_losses = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        ).view(shift_labels.size())

        mask        = (shift_labels != -100).float()
        denom       = mask.sum(dim=1).clamp(min=1)
        sample_loss = (token_losses * mask).sum(dim=1) / denom
        sample_score = -sample_loss

        for local_b, global_b in enumerate(valid_indices):
            scores[global_b][c] = sample_score[local_b].item()

    return [int(np.argmax(s)) for s in scores]

In [ ]:
# ── Quick sanity check — batch version (5 val examples) ──────────────────────
print("Sanity check — batched scoring on 5 val examples...")
model.eval()

batch_rows   = [val_df.iloc[i] for i in range(5)]
batch_images = [
    # Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    load_image(DATA_DIR / row["image_path"])
    for row in batch_rows
]

preds   = score_choices_loglik_batch(model, processor, batch_images, batch_rows, device)
correct = 0

for row, pred in zip(batch_rows, preds):
    gt   = int(row["answer"])
    mark = "✓" if pred == gt else "✗"
    print(f"  [{mark}] pred={CHOICE_LETTERS[pred]}  gt={CHOICE_LETTERS[gt]}  | {row['question']}")
    correct += int(pred == gt)

print(f"Zero-shot accuracy on 5 samples: {correct}/5")

In [ ]:
# ── Test Single inference loop ───────────────────────────────────────────────────────
import time

print("Running inference on test set...")
model.eval()
test_preds = []
SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"

t0 = time.time()

for i in tqdm(range(len(test_df)), desc="Test inference"):
    row   = test_df.iloc[i]
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    pred  = score_choices_loglik(model, processor, image, row, device)
    test_preds.append(pred)

# ── Build and save submission ──────────────────────────────────────────────────
submission = pd.DataFrame({
    "id":     test_df["id"].values,
    "answer": test_preds,
})

submission.to_csv(SUBMISSION_PATH, index=False)
elapsed_min = (time.time() - t0) / 60

print(f"\nSubmission saved to: {SUBMISSION_PATH}")
print(f"Runtime (minutes): {elapsed_min:.2f}")

submission.head()

In [ ]:
# ── Test Batched Inference + Submission Generation ───────────────────────────
# print("Running batched inference on test set...")
# model.eval()
# test_preds    = []
# SUBMISSION_PATH = OUTPUT_DIR / "submission.csv"

# t0 = time.time()
# INFER_BATCH = 8

# # Collect rows into mini-batches and process together
# for start in tqdm(range(0, len(test_df), INFER_BATCH), desc="Test inference (Batched)"):
#     batch_rows   = [test_df.iloc[i] for i in range(start, min(start + INFER_BATCH, len(test_df)))]
#     batch_images = [
#         Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
#         for row in batch_rows
#     ]
#     preds = score_choices_loglik_batch(model, processor, batch_images, batch_rows, device)
#     test_preds.extend(preds)

# # ── Build and save submission ─────────────────────────────────────────────────
# submission = pd.DataFrame({
#     "id":     test_df["id"].values,
#     "answer": test_preds,
# })

# submission.to_csv(SUBMISSION_PATH, index=False)
# elapsed_min = (time.time() - t0) / 60

# print(f"\nSubmission saved to: {SUBMISSION_PATH}")
# print(f"Runtime (minutes):   {elapsed_min:.2f}")
# print(f"Rows: {len(submission)} | Unique answers: {sorted(submission['answer'].unique())}")
# submission.head()

### **Section 11: Links**

> [Finals Report in ACL Format](https://drive.google.com/file/d/1nSo2DcXHcBo1jlaVpNXREW8AyKPuANgW/view?usp=sharing)

> [Model Weights in Hugging Face](https://huggingface.co/tk2558/DL_Finals_Model)

> [Github Repo](https://github.com/tk2558/Deep-Learning-Pixel-to-Predictions/tree/main)